# 01 · Data & preprocessing

Two stages sit in front of every model:

1. **Data** — a `MultiViewDataset` exposes `Xs` (per-view features), `As` (per-view adjacency), and `labels`.
2. **Preprocessing** — each view is *normalized* then *propagated* through its graph: `H = S · normalize(X)`.

This notebook shows the catalog API and walks the preprocessing pipeline one operation at a time.

## The dataset catalog

In [ ]:
from polyclustx.data import list_datasets, get_dataset_info

print('graph datasets   :', list_datasets(category='graph'))
print('feature datasets :', list_datasets(category='features')[:8], '...')
print('multimodal       :', list_datasets(category='multimodal')[:6], '...')

# Metadata without downloading:
get_dataset_info('acm')

Load one with `load_dataset(name)` (downloads + caches under `~/.polyclustx_datasets` on first use). Here we stay offline with the synthetic generator.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow

import numpy as np
import scipy.sparse as sp


def make_synthetic(n_per=60, k=3, dims=(30, 20), sep=1.0,
                   noise=(1.0, 2.5), p_noisy_edge=0.3, seed=0):
    """A small multi-view dataset with an uneven pair of views: (Xs, As, labels).

    Each view is Gaussian blobs around k centres. View 1 is deliberately
    noisier than view 0, and each graph mixes intra-cluster edges with a
    fraction of random ones -- so view weighting and propagation both have
    something real to do.
    """
    rng = np.random.RandomState(seed)
    n = n_per * k
    labels = np.repeat(np.arange(k), n_per)
    Xs, As = [], []
    for d, sigma in zip(dims, noise):
        centres = rng.randn(k, d) * sep
        Xs.append((centres[labels] + rng.randn(n, d) * sigma).astype(np.float64))
        A = np.zeros((n, n))
        for i in range(n):
            same = np.where(labels == labels[i])[0]
            A[i, rng.choice(same, size=5, replace=False)] = 1.0
            if rng.rand() < p_noisy_edge:
                A[i, rng.randint(n)] = 1.0          # a wrong edge
        As.append(sp.csr_matrix(np.maximum(A, A.T)))
    return Xs, As, labels


Xs, As, labels = make_synthetic()

# --- To use a real benchmark instead (downloads + caches on first call): ---
# from polyclustx.data import load_dataset
# data = load_dataset("acm")
# Xs, As, labels = data.Xs, data.As, data.labels

k = len(np.unique(labels))
print(f"views={len(Xs)}  samples={Xs[0].shape[0]}  clusters={k}")
print("view feature dims:", [X.shape[1] for X in Xs])

## The preprocessing pipeline

`get_propagated_features` is the single entry point the baseline uses per view. It runs, in order: optional column center/scale → row normalization (L2 or TF-IDF) → graph propagation `S·X` where `S = D^-1/2 (A + βI) D^-1/2`.

In [ ]:
from polyclustx.preprocessing import get_propagated_features, normalize_features, preprocess_graph

X0, A0 = Xs[0], As[0]
print('raw view 0        :', X0.shape, 'row norms ~', np.round(np.linalg.norm(X0, axis=1)[:3], 2))

# Stage A — normalize rows (project onto the unit sphere)
Xn = normalize_features(X0, tf_idf=False)
print('after L2 normalize:', np.round(np.linalg.norm(Xn, axis=1)[:3], 2))

# Stage B — build the propagation operator S from the adjacency
S = preprocess_graph(A0, beta=1.0)
print('propagation op S  :', S.shape, type(S).__name__)

# Full pipeline in one call (what the model does internally)
H = get_propagated_features(X0, adj=A0, beta=1.0, tf_idf=False)
print('propagated H      :', H.shape)

### Why propagation helps

Smoothing features over the graph pulls same-cluster nodes together. We can see it in the average intra- vs inter-cluster distance.

In [ ]:
def intra_inter(Z, y):
    from scipy.spatial.distance import cdist
    D = cdist(Z, Z)
    same = y[:, None] == y[None, :]
    return D[same].mean(), D[~same].mean()

for tag, Z in [('raw', X0), ('normalized', Xn), ('propagated', np.asarray(H))]:
    intra, inter = intra_inter(Z, labels)
    print(f'{tag:11s}  intra={intra:.3f}  inter={inter:.3f}  ratio={inter/intra:.2f}')

A higher inter/intra ratio means clusters are better separated — propagation should improve it. Per-view control (`beta`, `tf_idf`, `center`, `scale`) is exposed directly on the models.